# Task 1 — Preprocess and Explore the Data
**GMF Investments | Time Series Forecasting for Portfolio Management Optimization**

Objective: fetch historical data for TSLA, BND, and SPY (Jan 1, 2015 – Jun 30, 2026),
clean it, and run exploratory analysis (trend, volatility, stationarity, risk metrics)
to inform the forecasting work in later tasks.


In [ ]:
import sys
sys.path.append("..")  # allow importing from src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import (
    fetch_asset_data,
    combine_close_prices,
    clean_data,
    compute_daily_returns,
)
from src.eda import rolling_stats, detect_outliers, adf_test, print_adf_result
from src.risk_metrics import historical_var, sharpe_ratio

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

TICKERS = ["TSLA", "BND", "SPY"]
START, END = "2015-01-01", "2026-06-30"


## 1. Extract Historical Financial Data

Data is fetched via `yfinance` and cached to `data/raw/` as CSVs so repeated runs
don't re-hit the API. Delete the cached CSVs to force a fresh pull.


In [ ]:
raw_data = fetch_asset_data(tickers=TICKERS, start=START, end=END, cache_dir="../data/raw")

for ticker, df in raw_data.items():
    print(f"{ticker}: {df.shape[0]} rows, {df.index.min().date()} -> {df.index.max().date()}")
    display(df.head(3))


## 2. Data Cleaning and Understanding

- Combine the `Adj Close` price series for all three assets into one wide DataFrame.
- Check dtypes and basic statistics.
- Reindex to business-day frequency and fill gaps (holidays / missing feed days).


In [ ]:
prices = combine_close_prices(raw_data, price_col="Adj Close")
print("dtypes:")
print(prices.dtypes)
print()
print("Missing values before cleaning:")
print(prices.isna().sum())
prices.describe()


In [ ]:
prices_clean = clean_data(prices, method="ffill")
print("Missing values after cleaning:", prices_clean.isna().sum().sum())
print("Shape:", prices_clean.shape)
prices_clean.to_csv("../data/processed/adj_close_prices.csv")
prices_clean.tail()


**Data quality notes:** Yahoo Finance data may contain small gaps around exchange
holidays or feed outages. We reindex to a full business-day calendar and forward-fill
(then back-fill any leading gap) so every asset shares a common, continuous date index —
this is required before computing daily returns and running statistical tests. No
scaling/normalization is applied to price levels themselves; log/percentage returns
already inhabit a comparable scale for the downstream models, and standardization is
applied later, model-by-model, as `sklearn` scalers inside the LSTM pipeline (Task 2).


## 3. Exploratory Data Analysis (EDA)

### 3.1 Closing price over time


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for ticker in TICKERS:
    ax.plot(prices_clean.index, prices_clean[ticker], label=ticker)
ax.set_title("Adjusted Close Price (2015 – 2026)")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.legend()
plt.tight_layout()
plt.show()


### 3.2 Daily percentage change (volatility)

In [ ]:
returns = compute_daily_returns(prices_clean)
returns.to_csv("../data/processed/daily_returns.csv")

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for ax, ticker in zip(axes, TICKERS):
    ax.plot(returns.index, returns[ticker], linewidth=0.6, color="steelblue")
    ax.set_title(f"{ticker} Daily Return")
    ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

returns.describe()


### 3.3 Rolling mean & volatility (30-day window)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for ax, ticker in zip(axes, TICKERS):
    stats = rolling_stats(returns[ticker], window=30)
    ax.plot(stats.index, stats["rolling_std"], color="darkorange", label="30d rolling volatility")
    ax.set_title(f"{ticker} — Rolling 30-Day Volatility of Daily Returns")
    ax.legend()
plt.tight_layout()
plt.show()


### 3.4 Outlier detection

Days where a return moves more than 3 standard deviations from the mean.


In [ ]:
for ticker in TICKERS:
    outliers = detect_outliers(returns[ticker], n_std=3.0)
    print(f"\n{ticker}: {len(outliers)} outlier day(s) (>3 std)")
    if len(outliers):
        print(outliers.head(10))


## 4. Seasonality and Trend Analysis — Stationarity Testing

The Augmented Dickey-Fuller (ADF) test checks for a unit root. A high p-value means we
fail to reject the null hypothesis of non-stationarity — expected for raw price levels,
which trend over time. Daily returns, by contrast, should already be stationary, which
is what ARIMA needs (the 'd' order captures however much differencing is required to
get there).


In [ ]:
print("=== ADF Test: Price Levels ===")
for ticker in TICKERS:
    result = adf_test(prices_clean[ticker], name=f"{ticker} price")
    print_adf_result(result)
    print()


In [ ]:
print("=== ADF Test: Daily Returns ===")
for ticker in TICKERS:
    result = adf_test(returns[ticker], name=f"{ticker} return")
    print_adf_result(result)
    print()


**Interpretation:** if price levels come back non-stationary (typical) while
returns come back stationary (typical), this confirms `d=1` differencing is the right
starting point for ARIMA on TSLA prices — i.e., model the returns / first difference
rather than raw price levels directly.


## 5. Risk Metrics

- **Value at Risk (VaR)** — historical 95% VaR: the daily loss magnitude that returns
  are not expected to exceed 95% of the time.
- **Sharpe Ratio** — annualized risk-adjusted return, assuming a risk-free rate proxy
  (update `RISK_FREE_RATE` to the current T-bill yield as needed).


In [ ]:
RISK_FREE_RATE = 0.04  # annualized proxy; update to current risk-free rate as needed

risk_summary = pd.DataFrame({
    ticker: {
        "Historical VaR (95%, daily)": historical_var(returns[ticker], confidence=0.95),
        "Annualized Sharpe Ratio": sharpe_ratio(returns[ticker], risk_free_rate=RISK_FREE_RATE),
        "Mean Daily Return": returns[ticker].mean(),
        "Daily Volatility (std)": returns[ticker].std(),
    }
    for ticker in TICKERS
}).T

risk_summary


## 6. Key Insights (fill in after running against live data)

- **Tesla (TSLA):** Describe the overall price trajectory 2015–2026, and note the
  scale of daily-return volatility relative to BND/SPY.
- **Volatility clustering:** Note whether high-volatility periods (rolling std spikes)
  coincide with known market events (e.g., 2020 COVID crash, 2022 rate-hike cycle).
- **Outliers:** Summarize the largest single-day moves and, if identifiable, the
  events behind them.
- **Stationarity:** Confirm price series are non-stationary and returns are
  stationary — this determines the ARIMA differencing order used in Task 2.
- **Risk-adjusted return:** Compare Sharpe ratios across the three assets — expect
  BND to show the lowest volatility/VaR and SPY to sit between TSLA and BND on both
  risk and return.

*(This markdown cell is a template — replace with actual findings once the notebook
has been run against a live data pull outside this sandboxed environment.)*
